# 01 — Data Understanding

Customer Churn Prediction — Telco Dataset


## 1. Learning Objectives

By the end of this notebook you will be able to:

- Load a dataset through the project's own `src/` code instead of ad-hoc `pd.read_csv`.
- Describe a dataset's shape, column types, and target distribution using `shape`, `info()`, `describe()`, `isnull()`, `nunique()`, `value_counts()`, and `duplicated()`.
- Explain, in your own words, what each of those functions tells you and why it matters before you touch a model.
- Identify which columns are numerical vs categorical, and which single column must never be fed into a model.
- Recognize a "hidden" missing-value problem that `isnull()` alone cannot find.


## 2. Business Problem

> **Predict whether a customer is likely to churn (cancel their subscription), and understand which factors influence that decision.**

Before we can predict anything, we have to understand what we're working with. Every column in this dataset represents something a
telecom company already knows about a customer at the time they're still subscribed — their contract type, how they pay, what
services they've added, how long they've been a customer, and how much they're billed. The target we want to predict is the
`Churn` column: did this customer eventually leave?

This notebook does **no modeling**. Its only goal is to build an accurate mental model of the data before we clean it, engineer
features from it, or train anything on it. Skipping this step is one of the most common causes of confusing bugs later in a
project — you can't fix a problem in preprocessing if you don't know the problem exists yet.


## 3. Concept Explanation

**Why does "data understanding" come before EDA, cleaning, or modeling?**

A decision tree (or any model) doesn't know what a column *means* — it only sees numbers. Our job as the humans in the loop is to
catch problems a model would otherwise silently learn the wrong lesson from. Three concrete examples we'll hit in this exact
notebook:

1. **`customerID` is a unique identifier, not a feature.** If we accidentally left it in as a "category," a model with enough
   capacity could partially memorize which IDs churned in the training set — a textbook case of a feature that helps training
   accuracy while being completely useless (and actively misleading) on new customers.
2. **Not all missing values look like missing values.** `pandas.isnull()` only catches actual `NaN`. A column can be "secretly"
   missing data stored as blank text — which we'll prove below, not just assert.
3. **A numeric-looking column can be the wrong dtype.** `SeniorCitizen` is stored as `0`/`1`, which *looks* numerical, but it's
   really a yes/no category — treating it as a continuous number (e.g. scaling it) wouldn't be meaningful.

Catching these now means `03_preprocessing.ipynb` fixes them deliberately instead of a model quietly absorbing them.


## 4. Imports

We import the project's own `load_dataset` and `validate_dataset` functions rather than calling `pd.read_csv` directly in the
notebook. This is a deliberate professional-structure choice: the *logic* for loading and validating data lives once in
`src/data/`, and every notebook (and, later, any script or test) reuses the exact same logic instead of copy-pasting it. If we
ever need to change how loading works (e.g. read from a database instead of a CSV), we change it in one file.

We also add the project root to `sys.path`. Notebooks execute with their own working directory (`notebooks/`), which is one level
below the project root where the `src/` package lives — without this, `import src...` would fail with `ModuleNotFoundError`.


In [1]:
import sys
from pathlib import Path

# Make the project's src/ package importable from inside notebooks/.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config.settings import SETTINGS
from src.data.data_loader import load_dataset
from src.data.data_validator import validate_dataset

# Notebook display setting: show every column instead of truncating with "...".
# This is a pandas *display* option, unrelated to logging — it only affects
# how much of a DataFrame Jupyter prints, not the data itself.
pd.set_option("display.max_columns", None)


## 5. Load Data

Watch the output of this cell carefully: those lines prefixed with a timestamp and `INFO`/`WARNING` are not `print()` statements —
they're coming from the project's logger (`src/utils/logger.py`), and they're also being written to `logs/ml_project.log` at the
same time. In Jupyter they render as reddish text because Python's default logging handler writes to `stderr`, not `stdout` (this
is normal and not an error).

Notice the `WARNING` about `TotalCharges` — `data_validator.py` already found the hidden missing-value problem mentioned above,
automatically, before we've written a single line of exploration code ourselves.


In [2]:
dataframe = load_dataset(SETTINGS.data.raw_path)

expected_columns = [
    "customerID", "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges", "Churn",
]

validate_dataset(
    dataframe,
    expected_columns=expected_columns,
    target_column=SETTINGS.data.target_column,
)


2026-09-15 23:01:55 | INFO | src.data.data_loader | Loading dataset from C:\Users\DELL\OneDrive\Documents\PG-GEN-AI\decision-tree-learning\data\raw\telco_churn.csv


2026-09-15 23:01:55 | INFO | src.data.data_loader | Dataset loaded successfully. Shape: (7043, 21)


2026-09-15 23:01:55 | WARNING | src.data.data_validator | Column 'TotalCharges' looks numeric but is stored as text; 11 value(s) are blank/non-numeric rather than true NaN, so isnull() misses them. This needs explicit handling during preprocessing.


2026-09-15 23:01:55 | INFO | src.data.data_validator | Dataset validation passed: 7043 rows, 21 columns


## 6. Explore Data

### `shape`

`dataframe.shape` returns a `(rows, columns)` tuple. It's the very first thing to check: it tells you how much data you have
(enough to train a decision tree reliably?) and roughly how many features you'll be working with.


In [3]:
dataframe.shape

(7043, 21)

### `head()`

`head()` shows the first few rows so you can visually sanity-check that the data loaded the way you expect — column names line up
with values, nothing looks obviously shifted or corrupted.


In [4]:
dataframe.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### `info()`

`info()` lists every column with its dtype and non-null count in one place. It's the fastest way to spot two things at once:
columns with fewer non-null entries than the total row count (missing values), and columns whose dtype doesn't match what you'd
expect (e.g. a numeric-looking column stored as text — exactly `TotalCharges`'s problem).


In [5]:
dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

### `describe()`

By default, `describe()` only summarizes **numeric** columns: count, mean, std, min, the quartiles, and max. This is useful for
spotting impossible or suspicious values (e.g. a negative `tenure`, or a `MonthlyCharges` of 0) — we don't see any here, but this
is exactly the check that would catch them.


In [6]:
dataframe.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


Passing `include="object"` (or `include=["object", "string"]` on newer pandas, covered below) instead summarizes the
**categorical/text** columns: count, number of unique values, the most frequent value, and its frequency.


In [7]:
dataframe.describe(include=["object", "string"])

,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,TotalCharges,Churn
count,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043,7043
unique,7043,2,2,2,2,3,3,3,3,3,3,3,3,3,2,4,6531,2
top,7590-VHVEG,Male,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,20.2,No
freq,1,3555,3641,4933,6361,3390,3096,3498,3088,3095,3473,2810,2785,3875,4171,2365,11,5174


### `isnull()`

`dataframe.isnull()` returns a same-shaped DataFrame of `True`/`False`. Summed per column with `.sum()`, it tells you exactly how
many *true* `NaN` values each column has.

**This is the check that misses `TotalCharges`'s problem** — keep this number in mind and compare it with the coerced check two
cells down.


In [8]:
dataframe.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

### `nunique()`

`nunique()` counts how many *distinct* values each column has. This is how we tell numerical features (`tenure`,
`MonthlyCharges` — many unique values) apart from categorical ones (`Contract`, `PaymentMethod` — a handful of repeated
categories) *without* having to already know the schema by heart. It also flags `customerID` immediately: its unique count
should equal the row count, confirming it's a pure identifier.


In [9]:
dataframe.nunique().sort_values()

gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
PhoneService           2
PaperlessBilling       2
Churn                  2
MultipleLines          3
TechSupport            3
StreamingTV            3
OnlineBackup           3
DeviceProtection       3
StreamingMovies        3
Contract               3
OnlineSecurity         3
InternetService        3
PaymentMethod          4
tenure                73
MonthlyCharges      1585
TotalCharges        6531
customerID          7043
dtype: int64

In [10]:
assert dataframe["customerID"].nunique() == len(dataframe), "customerID should be unique per row"
print("customerID is a unique identifier for every one of the", len(dataframe), "rows.")

customerID is a unique identifier for every one of the 7043 rows.


### `value_counts()`

`value_counts()` counts how many rows fall into each distinct value of a single column — the natural next step after `nunique()`
tells you a column *has* only a few categories. Applied to our target column, this is also how we check for **class imbalance**.


In [11]:
dataframe["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [12]:
dataframe["Churn"].value_counts(normalize=True).mul(100).round(1)

Churn
No     73.5
Yes    26.5
Name: proportion, dtype: float64

### `duplicated()`

`dataframe.duplicated()` flags rows that are exact duplicates of an earlier row (every column matches). Summed, it gives a
duplicate row count. A high number here could mean the same customer's data was accidentally loaded twice, which would leak
information between train and test sets later if left unhandled.


In [13]:
dataframe.duplicated().sum()

np.int64(0)

## 7. Practical Implementation

Now that we've inspected the data manually, let's programmatically split the columns into numerical and categorical groups —
something `feature_engineering.py` and `preprocessor.py` will both need later. Rather than typing the schema out by hand (easy to
get wrong or forget to update), we derive it directly from the DataFrame's dtypes.


In [14]:
numerical_columns = dataframe.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_columns = dataframe.select_dtypes(include=["object", "string"]).columns.tolist()

print("Numerical columns:", numerical_columns)
print("Categorical columns:", categorical_columns)

Numerical columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges']
Categorical columns: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']


Notice `SeniorCitizen` landed in `numerical_columns` because it's stored as `0`/`1` — but conceptually it's a yes/no
category, not a continuous quantity. This is exactly the kind of thing dtype-based automation can get "technically right, but
semantically wrong," and why `03_preprocessing.ipynb` will treat it as categorical rather than trusting this automatic split
blindly.

Let's look at the value counts for a couple of categorical columns that intuitively feel like they'd relate to churn:


In [15]:
for column_name in ["Contract", "PaymentMethod", "InternetService"]:
    print(f"--- {column_name} ---")
    print(dataframe[column_name].value_counts())
    print()

--- Contract ---
Contract
Month-to-month    3875
Two year          1695
One year          1473
Name: count, dtype: int64

--- PaymentMethod ---
PaymentMethod
Electronic check             2365
Mailed check                 1612
Bank transfer (automatic)    1544
Credit card (automatic)      1522
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64



Now, the coerced check that catches what `isnull()` missed. We convert `TotalCharges` to numbers and let pandas turn
anything it can't parse into `NaN`, then compare that count against what `isnull()` reported above.


In [16]:
total_charges_true_missing = dataframe["TotalCharges"].isnull().sum()
total_charges_coerced = pd.to_numeric(dataframe["TotalCharges"], errors="coerce")
total_charges_hidden_missing = total_charges_coerced.isnull().sum()

print("Missing according to isnull():", total_charges_true_missing)
print("Missing after coercing to numeric:", total_charges_hidden_missing)

dataframe.loc[total_charges_coerced.isnull(), ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

Missing according to isnull(): 0
Missing after coercing to numeric: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,
753,3115-CZMZD,0,20.25,
936,5709-LVOEQ,0,80.85,
1082,4367-NUYAO,0,25.75,
1340,1371-DWPAZ,0,56.05,
3331,7644-OMVMY,0,19.85,
3826,3213-VVOLG,0,25.35,
4380,2520-SGTTA,0,20.00,
5218,2923-ARZLG,0,19.70,
6670,4075-WKNIU,0,73.35,


## 8. Results

- **Shape:** 7,043 rows × 21 columns.
- **Target column:** `Churn`, with **5,174 "No" (73.5%) vs. 1,869 "Yes" (26.5%)** — a real but moderate class imbalance we must
  account for during evaluation (Section 19 of the project spec).
- **`isnull()` found 0 missing values anywhere** — but coercing `TotalCharges` to numeric reveals **11 rows** with a blank string
  instead of a number. Every one of those 11 rows has `tenure == 0`: these are brand-new customers who haven't been billed a
  full cycle yet, so `TotalCharges` was never populated.
- **`customerID` is unique for all 7,043 rows** — confirmed to be a pure identifier that must be dropped before modeling.
- **0 fully duplicated rows.**
- 3 numerical columns by dtype (`SeniorCitizen`, `tenure`, `MonthlyCharges`) plus `TotalCharges` once cleaned; the remaining
  17 columns are categorical (including `SeniorCitizen` semantically, and excluding `customerID`).


## 9. Interpretation

- The 26.5% churn rate means a model that **always predicts "No churn"** would already be about 73.5% accurate — that number
  will become our baseline in `05_model_evaluation.ipynb`, and it's why **accuracy alone will not be a trustworthy metric** for
  this problem.
- The 11 `TotalCharges` rows aren't random noise — they follow a clear rule (`tenure == 0`). This means the "right" fix is not to
  drop them or fill them with the column mean (which would misrepresent brand-new customers as average spenders), but to fill
  them with `0`, matching the business reality that a customer with zero tenure has been billed zero dollars so far. We'll do
  this explicitly, with a comment explaining why, in `data_cleaner.py`.
- `customerID` will be dropped (not encoded) in preprocessing — it carries no predictive signal.
- Because categorical columns vastly outnumber numerical ones, encoding strategy will be a central decision in
  `03_preprocessing.ipynb`.


## 10. Common Mistakes

- **Trusting `isnull().sum() == 0` as proof of "no missing data."** We just showed it isn't — always also check for blank
  strings, placeholder values like `"?"`, `"N/A"`, or `-1` in numeric columns, and columns typed as text that "look" numeric.
- **Feeding `customerID` into a model "just in case."** A tree-based model can and will find spurious splits on an identifier if
  it's present, which inflates training accuracy while learning nothing generalizable.
- **Assuming `select_dtypes` gives you the semantically correct feature grouping.** As seen with `SeniorCitizen`, dtype-based
  splitting is a starting point, not a final answer — always sanity-check it against what each column *means*.
- **Skipping `dataframe.head()`.** It feels redundant next to `info()`/`describe()`, but it's often the fastest way to notice a
  column got shifted, a delimiter was parsed wrong, or a value looks encoded differently than you expected.


## 11. What We Learned

- How to load and validate data through the project's own reusable `src/` code instead of notebook-only throwaway code.
- What `shape`, `head()`, `info()`, `describe()`, `isnull()`, `nunique()`, `value_counts()`, and `duplicated()` each reveal, and
  in what order it makes sense to run them.
- That `Churn` is imbalanced (~26.5% positive class), which will shape how we evaluate models later.
- That `TotalCharges` has 11 hidden missing values, tied to `tenure == 0`, invisible to `isnull()` alone.
- That `customerID` is a unique identifier that must never be used as a model feature.


## 12. Exercises

Try these directly in this notebook (add cells below) before moving on to `02_eda.ipynb`:

1. Use `value_counts()` on `gender` and on `SeniorCitizen`. Are they balanced? Does an imbalanced *feature* (as opposed to an
   imbalanced *target*) cause the same problems? Why or why not?
2. Run `dataframe.describe(include="all")` (note: plain `"all"`, not the object-only list we used above) and find one number in
   the output that only makes sense for numeric columns and one that only makes sense for categorical columns.
3. `MonthlyCharges` is numeric and `TotalCharges` should be too, once cleaned. Compute `dataframe["MonthlyCharges"] * dataframe["tenure"]`
   for a few rows and compare it to `TotalCharges`. Are they close? What might explain small differences (hint: pricing changes
   over time, promotions, mid-cycle plan switches)?
4. **Interview question:** Why is it dangerous to fill a missing numeric value with the column's mean *before* you've understood
   *why* the value is missing?
5. **Interview question:** `dataframe.isnull().sum()` reported zero missing values, yet we still found a real data quality
   problem. What general lesson does this teach about trusting a single diagnostic function?
